In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
import joblib

model = joblib.load("rf_model_precision1.joblib")
scaler = joblib.load("rf_model_precision1_scaler.joblib")

In [4]:
model, scaler

(RandomForestClassifier(class_weight='balanced'), StandardScaler())

In [5]:
scaler.feature_names_in_

array(['나이', '참여프로젝트', '월급', '이직회수', '주변평가', '경력', '전년도교육출장횟수', '현회사근속년수',
       '출장_등급', '이직률', '프로젝트참여율', '교육출장참여율', '현직근속비율', '연봉/경력비율',
       '연봉/프로젝트비율', '경력-근속연차', '근속연차차이', '프로젝트밀도지수', '평판×근속연차', '연봉×평판점수',
       '경력/나이비율', '근속/나이비율', '연봉/나이', '입사나이', '출장odm'], dtype=object)

In [6]:
#'나이', '참여프로젝트', '월급', '이직회수', '주변평가', '경력', '전년도교육출장횟수', '현회사근속년수',
# 출장 등급
# 이직률 = 이직수 / 경력
# 프로젝트참여율 = 참여프로젝트 / 경력
# 교육출장참여율 = 전년도교육출장횟수 / 경력
# 현직근속비율 = 현회사근속연수 / 경력
# 연봉/경력비율 = 월급*12/경력
# 연봉/프로젝트비율 = 월급*12/참여프로젝트
# 경력-근속연차 = 경력-근속연차
# 근속연차나이 = 근속연차 - 현회사근속연수
# 프로젝트밀도지수 = 참여프로젝트+전년도교육출장횟수 / 경력
# 평판x근속년수 = 주변평가 * 현회사근속년수
# 연봉x평판점수 = 월급*12*주변평가
# 경력/나이비율 = 경력/나이
# 근속/나이비율 = 현회사근속년수/나이
# 연봉/나이 = 월급*12/나이
# 입사나이 = 나이 - 경력
# 출장 odm

# 사용자한테 받을 값 
- 나이 : int
- 참여프로젝트 : int
- 월급 : int
- 이직회수 : int
- 주변평가 : int [1,2,3,4]
- 경력 : int
- 전년도교육출장횟수 : int
- 현회사근속년수 : int
- 현부서근속연차 : int
- 출장 횟수 : n 회 -> if 분기로 0,1,2 구분 필요

```
array(['나이', '참여프로젝트', '월급', '이직회수', '주변평가', '경력', '전년도교육출장횟수', '현회사근속년수',
       '출장_등급', '이직률', '프로젝트참여율', '교육출장참여율', '현직근속비율', '연봉/경력비율',
       '연봉/프로젝트비율', '경력-근속연차', '근속연차차이', '프로젝트밀도지수', '평판×근속연차', '연봉×평판점수',
       '경력/나이비율', '근속/나이비율', '연봉/나이', '입사나이', '출장odm'], dtype=object)
```

In [40]:
def to_xcol(age, project, salary, number_of_turnovers, surround_eval, personal_history, edu_trips_lastyear, currentyear_at_company,currentyear_at_depart, buisnesstrip,when_enroll) :
    '''
    [나이, 참여프로젝트 수, 월급, 이직횟수, 주변평가(1~4),경력,
    전년도교육출장횟수, 현회사근속년수, 현부서근속년수, 출장횟수, 입사나이]
    를 입력받아 RandomForestClassifier에 입력할 값으로 변환 후 반환하는 함수
    '''
    import joblib
    # 필요 model, scaler import
    model = joblib.load("rf_model_precision1.joblib")
    scaler = joblib.load("rf_model_precision1_scaler.joblib")
    
    나이=age; 참여프로젝트=project; 월급=salary; 이직회수=number_of_turnovers; 주변평가=surround_eval;
    경력=personal_history; 전년도교육출장횟수=edu_trips_lastyear; 현회사근속년수=currentyear_at_company
    현부서근속년수=currentyear_at_depart; 출장횟수=buisnesstrip ; 입사나이 = when_enroll
    출장_등급 = 0 if 출장횟수==0 else 1 if 1<=출장횟수<=29 else 2
    근속연차 = 현회사근속년수 - 입사나이
    이직률 = 이직회수 / 경력
    프로젝트참여율 = 참여프로젝트 / 경력
    교육출장참여율 = 전년도교육출장횟수 / 경력
    현직근속비율 = 현회사근속년수 / 경력
    연봉_경력비율 = 월급*12/경력
    연봉_프로젝트비율 = 월급*12/참여프로젝트
    경력_근속연차 = 경력-근속연차
    근속연차차이 = 근속연차 - 현회사근속년수
    프로젝트밀도지수 = 참여프로젝트+전년도교육출장횟수 / 경력
    평판_근속년수 = 주변평가 * 현회사근속년수
    연봉_평판점수 = 월급*12*주변평가
    경력_나이비율 = 경력/나이
    근속_나이비율 = 현회사근속년수/나이
    연봉_나이 = 월급*12/나이
    입사나이 = 나이 - 경력
    출장odm = 출장_등급
    x_col = [[나이, 참여프로젝트, 월급, 이직회수, 주변평가, 경력, 전년도교육출장횟수, 현회사근속년수,출장_등급, 이직률, 프로젝트참여율, 교육출장참여율, 현직근속비율, 연봉_경력비율,연봉_프로젝트비율, 경력-근속연차, 근속연차차이, 프로젝트밀도지수, 평판_근속년수, 연봉_평판점수,경력_나이비율, 근속_나이비율, 연봉_나이, 입사나이, 출장odm]]
    result = model.predict(scaler.transform(x_col))
    return "우수" if result != 0 else "보통"

In [41]:
to_xcol(30,2,3000000,0,2,1,2,1,1,2,30)

C:\Users\Admin\anaconda3\envs\ml-dl-nlp\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


'보통'

In [32]:
model.predict(to_xcol(30,2,3000000,0,2,1,2,1,1,2,30))

array([0], dtype=int64)